# Stride-8 aligned retrieval revalidation — self-contained v4

This version removes the fragile assumption that historical Exp33/Exp35B/Exp36R
artifacts must live under one project root.

**Inputs required from the server**
- the existing Exp33 frozen manifests,
- the six raw benchmark datasets,
- any stride-8 caches already produced by v2/v3 (automatically reused).

**Embedded frozen reference**
- the original 24-condition Pearson / PRR-Stat / full-PRR table from the
  supplementary material is embedded directly in this notebook.
- therefore `results/exp36R/...` and `results/exp35B/...` are no longer required.

**Revalidation**
- rerun PRR-Stat only for Weather / Electricity / Traffic / Solar,
- reuse ETTh1 / Exchange PRR-Stat because they were already stride 8,
- recompute Pearson for all 24 conditions with stride 8,
- keep the frozen full-PRR values,
- rebuild the three-way retrieval table and headline statistics.

In [1]:
# Cell 1 — stride-8 revalidation configuration (self-contained v4)
#
# Key change from v3:
#   - historical Exp36R / Exp35B directories are NOT required.
#   - the frozen 24-condition reference table from the supplementary material
#     is embedded below.
#   - manifests, datasets, and any already-created stride=8 caches are located
#     independently.

from pathlib import Path
import os, json, math, random, time, gc, io
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

# -------------------------------------------------------------------------
# 1) Frozen reference table embedded from supplementary material
# -------------------------------------------------------------------------
REFERENCE_CSV = r"""dataset,horizon,pearson_analog_future_mse,prr_stat_analog_future_mse,prr_analog_future_mse
ETTh1,96,1.6123,1.1965,1.1643
ETTh1,192,1.8175,1.3664,1.2229
ETTh1,336,1.9929,1.4861,1.4816
ETTh1,720,2.2541,1.7272,1.4618
Weather,96,0.8462,0.7558,0.4316
Weather,192,0.8782,0.7419,0.5147
Weather,336,1.2565,0.9200,0.6748
Weather,720,1.2669,1.1345,0.8130
Electricity,96,2.6259,1.9919,0.7191
Electricity,192,2.6210,2.0097,0.7345
Electricity,336,2.6610,2.0390,0.7648
Electricity,720,2.8247,2.1850,0.8602
Traffic,96,4.3063,3.5581,1.4696
Traffic,192,4.4209,3.8085,1.4508
Traffic,336,4.4387,3.8961,1.5147
Traffic,720,4.6757,3.9993,1.7351
Exchange,96,0.2133,0.2206,0.1652
Exchange,192,0.4234,0.4276,0.4116
Exchange,336,0.7643,0.7714,0.8094
Exchange,720,1.8868,2.1310,1.7770
Solar,96,0.7709,0.6046,0.5087
Solar,192,0.7623,0.6381,0.5406
Solar,336,0.7939,0.6707,0.5701
Solar,720,0.8067,0.6961,0.5507
"""
REFERENCE24 = pd.read_csv(io.StringIO(REFERENCE_CSV))
REFERENCE24["horizon"] = REFERENCE24["horizon"].astype(int)
assert len(REFERENCE24) == 24

# -------------------------------------------------------------------------
# 2) Locate Exp33 manifests independently
# -------------------------------------------------------------------------
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

manifest_candidates = [
    Path("/data/code/which_histories_matter_ext/results/exp33/final_eval_manifest"),
    Path("/data/code/stock_regime_retrieval/strong_forecaster/results/exp33/final_eval_manifest"),
]

# Add shallow glob discovery.
for base in [Path("/data/code"), Path("/data")]:
    if base.exists():
        try:
            manifest_candidates += list(base.glob("*/results/exp33/final_eval_manifest"))
            manifest_candidates += list(base.glob("*/*/results/exp33/final_eval_manifest"))
        except Exception:
            pass

MANIFEST_ROOT = first_existing(manifest_candidates)
if MANIFEST_ROOT is None:
    raise FileNotFoundError(
        "Could not locate results/exp33/final_eval_manifest under /data/code or /data."
    )

# The project root that owns the manifests is only used as a default output root.
PROJECT_ROOT = MANIFEST_ROOT.parents[2]

# -------------------------------------------------------------------------
# 3) Reuse any stride=8 cache created by v2/v3, regardless of project root
# -------------------------------------------------------------------------
reval_candidates = [
    Path("/data/code/which_histories_matter_ext/results/stride8_revalidation"),
    Path("/data/code/stock_regime_retrieval/strong_forecaster/results/stride8_revalidation"),
    PROJECT_ROOT / "results" / "stride8_revalidation",
]

for base in [Path("/data/code"), Path("/data")]:
    if base.exists():
        try:
            reval_candidates += list(base.glob("*/results/stride8_revalidation"))
            reval_candidates += list(base.glob("*/*/results/stride8_revalidation"))
        except Exception:
            pass

def cache_score(p):
    p = Path(p)
    if not p.exists():
        return -1
    # Prefer the directory with the most already-completed PRR-Stat condition JSONs.
    return len(list((p / "prr_stat").glob("*_result.json"))) if (p / "prr_stat").exists() else 0

existing_reval = [Path(p) for p in reval_candidates if Path(p).exists()]
if existing_reval:
    REVAL_ROOT = max(existing_reval, key=cache_score)
else:
    REVAL_ROOT = PROJECT_ROOT / "results" / "stride8_revalidation"

# ROOT is kept only for backward compatibility with the copied implementation cells.
ROOT = PROJECT_ROOT

PRRSTAT_ROOT = REVAL_ROOT / "prr_stat"
PEARSON_ROOT = REVAL_ROOT / "pearson"
CKPT_ROOT = PRRSTAT_ROOT / "checkpoints"
PRRSTAT_CACHE_ROOT = PRRSTAT_ROOT / "retrieval_cache"
PEARSON_CACHE_ROOT = PEARSON_ROOT / "test_cache"

for p in [REVAL_ROOT, PRRSTAT_ROOT, PEARSON_ROOT, CKPT_ROOT,
          PRRSTAT_CACHE_ROOT, PEARSON_CACHE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)
CHANGED_DATASETS = ("Weather","Electricity","Traffic","Solar")
UNCHANGED_DATASETS = ("ETTh1","Exchange")

CHANNELS = {
    "ETTh1":7,
    "Weather":21,
    "Electricity":321,
    "Traffic":862,
    "Exchange":8,
    "Solar":137,
}

RERUN_DATASETS = CHANGED_DATASETS
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class CFG:
    retrieval_seq_len = 96
    top_m = 100
    top_k = 10

    lr = 1e-3
    wd = 1e-4
    batch = 128
    max_epochs = 30
    patience = 6
    grad_clip = 5.0
    target_tau = 0.5
    feature_clip = 8.0

    initial_memory_fraction = 0.60
    train_cap_mechanism = 6000
    train_cap_frozen = 10000
    seed = 0

cfg = CFG()
EPS = 1e-8

print("MANIFEST_ROOT:", MANIFEST_ROOT)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("REVAL_ROOT   :", REVAL_ROOT)
print("Cached PRR-Stat condition JSONs:", cache_score(REVAL_ROOT))
print("DEVICE       :", DEVICE)
print("[PASS] v4 configuration; no Exp36R/Exp35B dependency")

MANIFEST_ROOT: /data/code/which_histories_matter_ext/results/exp33/final_eval_manifest
PROJECT_ROOT : /data/code/which_histories_matter_ext
REVAL_ROOT   : /data/code/which_histories_matter_ext/results/stride8_revalidation
Cached PRR-Stat condition JSONs: 16
DEVICE       : cuda:0
[PASS] v4 configuration; no Exp36R/Exp35B dependency


## 1. Exact PRR-Stat implementation

In [2]:

# Cell 2 — exact PRR-Stat architecture

def set_seed(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))

def inverse_softplus(x):
    return math.log(math.exp(float(x))-1.0)

class PRRStatReranker(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(29,128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128,128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128,1),
        )
        self.raw_alpha = nn.Parameter(
            torch.tensor(
                inverse_softplus(0.1),
                dtype=torch.float32,
            )
        )

    def forward(self,feat,base):
        delta=self.net(feat).squeeze(-1)
        alpha=F.softplus(self.raw_alpha)
        return base+alpha*delta

print("PRR-Stat input dim =",29)
print("Parameters =",sum(p.numel() for p in PRRStatReranker().parameters()))


PRR-Stat input dim = 29
Parameters = 20994


In [3]:

# Cell 3 — datasets, train-only normalization, protocol boundaries

DATA_CANDIDATES = {
    "ETTh1":[
        Path("/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv"),
        Path("/data/dataset/ETTh1.csv"),
    ],
    "Weather":[
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path("/data/dataset/weather.csv"),
    ],
    "Electricity":[
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity.csv"),
    ],
    "Traffic":[
        Path("/data/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/Time-Series-Library_v2/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic.csv"),
    ],
    "Exchange":[
        Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate.csv"),
        Path("/data/dataset/exchange.csv"),
    ],
    "Solar":[
        Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/solar_AL.txt"),
    ],
}

def read_raw(dataset):
    p=next((p for p in DATA_CANDIDATES[dataset] if p.exists()),None)
    assert p is not None,dataset

    if dataset=="Solar":
        try:
            df=pd.read_csv(p,header=None)
            if df.shape[1]==1:
                df=pd.read_csv(p,header=None,sep=r"\s+")
        except Exception:
            df=pd.read_csv(p,header=None,sep=r"\s+")
    else:
        df=pd.read_csv(p)
        if "date" in df.columns:
            df=df.drop(columns=["date"])
        if "OT" in df.columns:
            df=df[[x for x in df.columns if x!="OT"]+["OT"]]

    df=(
        df.apply(pd.to_numeric,errors="coerce")
        .dropna(axis=1,how="all")
        .replace([np.inf,-np.inf],np.nan)
        .interpolate(axis=0,limit_direction="both")
        .ffill()
        .bfill()
    )
    return df.to_numpy(np.float32)

def split_bounds(dataset,n):
    if dataset=="ETTh1":
        train_end=12*30*24
        val_end=train_end+4*30*24
    else:
        train_end=int(.70*n)
        val_end=n-int(.20*n)
    return train_end,val_end,n

def memory_stride(dataset):
    # Revalidation contract: ALL six datasets use stride=8.
    return 8

DATA={}

for dataset in DATASETS:
    raw=read_raw(dataset)
    assert raw.shape[1]==CHANNELS[dataset],(dataset,raw.shape)

    tr,va,te=split_bounds(dataset,len(raw))

    mu=raw[:tr].mean(axis=0).astype(np.float32)
    sd=raw[:tr].std(axis=0,ddof=0).astype(np.float32)
    assert np.all(sd>1e-6)

    z=((raw-mu[None,:])/sd[None,:]).astype(np.float32)

    DATA[dataset]=dict(
        z=z,
        train_end=tr,
        val_end=va,
        test_end=te,
    )

    print(dataset,raw.shape,"train",tr,"val",va)

print("[PASS] six datasets ready")


ETTh1 (17420, 7) train 8640 val 11520
Weather (52696, 21) train 36887 val 42157
Electricity (26304, 321) train 18412 val 21044
Traffic (17544, 862) train 12280 val 14036
Exchange (7588, 8) train 5311 val 6071
Solar (52560, 137) train 36792 val 42048
[PASS] six datasets ready


In [4]:

# Cell 4 — observable features and deterministic pair sampling

def batch_pattern(x):
    x=np.asarray(x,np.float32)
    xc=x-x.mean(axis=-1,keepdims=True)
    n=np.linalg.norm(xc,axis=-1,keepdims=True)
    return np.where(
        n>EPS,
        xc/np.maximum(n,EPS),
        0.0,
    ).astype(np.float32)

def context7(x):
    x=np.asarray(x,np.float32)
    short=max(8,cfg.retrieval_seq_len//4)

    m=x.mean(axis=-1)
    s=x.std(axis=-1)+EPS

    f1=(x[...,-1]-m)/s
    f2=(x[...,-short:].mean(axis=-1)-m)/s
    f3=(x[...,-1]-x[...,-short])/s
    f4=(x[...,-1]-x[...,0])/s

    dx=np.diff(x,axis=-1)
    ds=np.diff(x[...,-short:],axis=-1)
    f5=(ds.std(axis=-1)+EPS)/(dx.std(axis=-1)+EPS)

    t=np.linspace(-1.,1.,cfg.retrieval_seq_len,dtype=np.float32)
    t=t-t.mean()
    f6=(
        np.sum(t*(x-m[...,None]),axis=-1)
        /(np.sum(t*t)+EPS)
    )/s

    a=x[...,:-1]
    b=x[...,1:]
    a=a-a.mean(axis=-1,keepdims=True)
    b=b-b.mean(axis=-1,keepdims=True)
    f7=np.sum(a*b,axis=-1)/(
        np.sqrt(
            np.sum(a*a,axis=-1)
            *np.sum(b*b,axis=-1)
        )+EPS
    )

    return np.stack(
        [f1,f2,f3,f4,f5,f6,f7],
        axis=-1,
    ).astype(np.float32)

def past_windows(z,ch,anchors):
    anchors=np.asarray(anchors,np.int64)
    idx=(
        anchors[:,None]
        -cfg.retrieval_seq_len
        +np.arange(cfg.retrieval_seq_len)[None,:]
    )
    return z[idx,ch].astype(np.float32)

def future_residuals(z,ch,anchors,h):
    anchors=np.asarray(anchors,np.int64)
    idx=anchors[:,None]+np.arange(h)[None,:]
    future=z[idx,ch].astype(np.float32)
    current=z[anchors-1,ch].astype(np.float32)
    return (future-current[:,None]).astype(np.float32)

def balanced_pairs(dataset,anchors,n_pairs,seed):
    anchors=np.asarray(anchors,np.int64)
    C=CHANNELS[dataset]

    n_pairs=min(int(n_pairs),len(anchors)*C)
    base=n_pairs//C
    rem=n_pairs%C

    rng=np.random.default_rng(seed)
    extras=set(
        rng.choice(C,size=rem,replace=False).tolist()
        if rem else []
    )

    aa=[]
    cc=[]

    for ch in range(C):
        n=base+(1 if ch in extras else 0)
        if n<=0:
            continue
        sel=rng.choice(
            anchors,
            size=n,
            replace=(n>len(anchors)),
        )
        aa.append(sel.astype(np.int64))
        cc.append(np.full(n,ch,np.int64))

    a=np.concatenate(aa)
    c=np.concatenate(cc)

    order=np.lexsort((c,a))
    return a[order],c[order]

print("[PASS] feature/sampling helpers ready")


[PASS] feature/sampling helpers ready


In [5]:
# Cell 5 — exact frozen validation/test manifests

def load_manifest(dataset,h,split):
    p = MANIFEST_ROOT / f"{dataset}_H{h}_{split}.npz"
    assert p.exists(), p
    with np.load(p, allow_pickle=False) as z:
        a = np.asarray(z["anchor"], np.int64)
        c = np.asarray(z["channel"], np.int64)
    assert a.shape == c.shape
    assert len(a) > 0
    return a,c

audit = []
for dataset in DATASETS:
    for h in HORIZONS:
        for split in ("val","test"):
            a,c = load_manifest(dataset,h,split)
            audit.append({
                "dataset":dataset, "horizon":h, "split":split,
                "pairs":len(a), "channels":len(np.unique(c)),
                "anchor_min":int(a.min()), "anchor_max":int(a.max())
            })

manifest_audit = pd.DataFrame(audit)
display(manifest_audit)
assert len(manifest_audit) == 48
print("[PASS] exact frozen manifests = 48/48")

,dataset,horizon,split,pairs,channels,anchor_min,anchor_max
0,ETTh1,96,val,4096,7,8640,11424
1,ETTh1,96,test,8192,7,11520,17323
2,ETTh1,192,val,4096,7,8640,11328
3,ETTh1,192,test,8192,7,11520,17228
4,ETTh1,336,val,4096,7,8640,11184
5,ETTh1,336,test,8192,7,11520,17084
6,ETTh1,720,val,4096,7,8640,10800
7,ETTh1,720,test,8192,7,11520,16700
8,Weather,96,val,4096,21,36887,42061
9,Weather,96,test,8192,21,42159,52600


[PASS] exact frozen manifests = 48/48


In [6]:

# Cell 6 — build PRR-Stat scaler from the initial training memory

def fit_feature_scaler(dataset,h):
    z=DATA[dataset]["z"]
    train_end=DATA[dataset]["train_end"]
    boundary=int(cfg.initial_memory_fraction*train_end)

    stride=memory_stride(dataset)

    anchors=np.arange(
        cfg.retrieval_seq_len,
        boundary-h+1,
        stride,
        dtype=np.int64,
    )

    all_stats=[]

    for ch in range(CHANNELS[dataset]):
        if len(anchors)==0:
            continue
        p=past_windows(z,ch,anchors)
        all_stats.append(context7(p))

    x=np.concatenate(all_stats,axis=0)
    med=np.median(x,axis=0).astype(np.float32)
    iqr=(
        np.percentile(x,75,axis=0)
        -np.percentile(x,25,axis=0)
    ).astype(np.float32)
    iqr=np.where(iqr<1e-5,1.0,iqr).astype(np.float32)

    return med,iqr

def scale_stats(x,med,iqr):
    return np.clip(
        (x-med)/iqr,
        -cfg.feature_clip,
        cfg.feature_clip,
    ).astype(np.float32)

print("[PASS] scaler helpers ready")


[PASS] scaler helpers ready


In [7]:

# Cell 7 — construct listwise PRR-Stat supervision

def top_m_indices(scores,m):
    m=min(int(m),len(scores))
    part=np.argpartition(-scores,m-1)[:m]
    return part[np.argsort(-scores[part])]

def build_supervision(
    dataset,
    h,
    query_anchor,
    query_channel,
    memory_boundary,
    med,
    iqr,
    need_candidate_future=False,
):
    """
    Returns query-wise Pattern Top-M features and future-supervised targets.

    feat: [N,M,29]
    base: [N,M]
    target: [N,M]
    distance: [N,M]
    candidate_anchor: [N,M]
    """
    z=DATA[dataset]["z"]
    N=len(query_anchor)
    M=cfg.top_m

    feat=np.empty((N,M,29),np.float32)
    base=np.empty((N,M),np.float32)
    target=np.empty((N,M),np.float32)
    dist=np.empty((N,M),np.float32)
    cand_anchor=np.empty((N,M),np.int64)

    stride=memory_stride(dataset)

    for ch in np.unique(query_channel):
        rows=np.flatnonzero(query_channel==ch)
        qa=query_anchor[rows]

        mem_a=np.arange(
            cfg.retrieval_seq_len,
            int(memory_boundary)-h+1,
            stride,
            dtype=np.int64,
        )

        if len(mem_a)<M:
            raise ValueError(
                f"{dataset} H={h} ch={ch}: memory={len(mem_a)} < M={M}"
            )

        mem_past=past_windows(z,int(ch),mem_a)
        mem_pat=batch_pattern(mem_past)
        mem_stat=scale_stats(
            context7(mem_past),
            med,
            iqr,
        )
        mem_future=future_residuals(
            z,int(ch),mem_a,h
        )

        q_past=past_windows(z,int(ch),qa)
        q_pat=batch_pattern(q_past)
        q_stat=scale_stats(
            context7(q_past),
            med,
            iqr,
        )
        q_future=future_residuals(
            z,int(ch),qa,h
        )

        ps=q_pat@mem_pat.T

        for local,row in enumerate(rows):
            ix=top_m_indices(ps[local],M)

            b=ps[local,ix].astype(np.float32)
            cs=mem_stat[ix]
            qs=q_stat[local]

            qrep=np.repeat(
                qs[None,:],
                M,
                axis=0,
            )

            f=np.concatenate(
                [
                    b[:,None],
                    qrep,
                    cs,
                    qrep-cs,
                    np.abs(qrep-cs),
                ],
                axis=1,
            ).astype(np.float32)

            assert f.shape==(M,29)

            dd=np.mean(
                (
                    mem_future[ix]
                    -q_future[local][None,:]
                )**2,
                axis=1,
            ).astype(np.float32)

            mu=float(dd.mean())
            sd=float(dd.std())
            zdist=(dd-mu)/max(sd,1e-6)

            logits=-zdist/cfg.target_tau
            logits=logits-logits.max()
            p=np.exp(logits)
            p=(p/p.sum()).astype(np.float32)

            feat[row]=f
            base[row]=b
            target[row]=p
            dist[row]=dd
            cand_anchor[row]=mem_a[ix]

    out=dict(
        feat=feat,
        base=base,
        target=target,
        distance=dist,
        candidate_anchor=cand_anchor,
        anchor=np.asarray(query_anchor,np.int64),
        channel=np.asarray(query_channel,np.int64),
    )

    return out

print("[PASS] listwise supervision builder ready")


[PASS] listwise supervision builder ready


In [8]:

# Cell 8 — training / validation functions

def listwise_epoch(model,opt,data,rng):
    model.train()

    n=len(data["feat"])
    order=rng.permutation(n)

    total=0.0
    count=0

    for left in range(0,n,cfg.batch):
        ix=order[left:left+cfg.batch]

        feat=torch.from_numpy(
            data["feat"][ix]
        ).float().to(DEVICE)

        base=torch.from_numpy(
            data["base"][ix]
        ).float().to(DEVICE)

        target=torch.from_numpy(
            data["target"][ix]
        ).float().to(DEVICE)

        score=model(feat,base)
        logp=F.log_softmax(score,dim=1)

        loss=-(target*logp).sum(dim=1).mean()

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            cfg.grad_clip,
        )
        opt.step()

        total+=float(loss.detach())*len(ix)
        count+=len(ix)

    return total/max(count,1)

@torch.no_grad()
def analog_mse(model,data):
    model.eval()

    vals=[]

    for left in range(0,len(data["feat"]),256):
        right=min(len(data["feat"]),left+256)

        feat=torch.from_numpy(
            data["feat"][left:right]
        ).float().to(DEVICE)

        base=torch.from_numpy(
            data["base"][left:right]
        ).float().to(DEVICE)

        score=model(feat,base).cpu().numpy()

        top=np.argsort(
            -score,
            axis=1,
        )[:,:cfg.top_k]

        d=np.take_along_axis(
            data["distance"][left:right],
            top,
            axis=1,
        )

        vals.append(d.mean(axis=1))

    return float(np.concatenate(vals).mean())

def fit_phase_a(train_data,val_data,seed):
    set_seed(seed)

    model=PRRStatReranker().to(DEVICE)
    opt=torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.wd,
    )

    rng=np.random.default_rng(seed+101)

    best=np.inf
    best_epoch=1
    best_state=None
    wait=0
    history=[]

    for epoch in range(1,cfg.max_epochs+1):
        train_loss=listwise_epoch(
            model,opt,train_data,rng
        )

        val_metric=analog_mse(
            model,val_data
        )

        history.append({
            "epoch":epoch,
            "train_fc_loss":train_loss,
            "val_analog_mse":val_metric,
            "alpha":float(
                F.softplus(
                    model.raw_alpha
                ).detach().cpu()
            ),
        })

        print(
            f"  ep={epoch:02d} "
            f"FC={train_loss:.5f} "
            f"valAnalog={val_metric:.6f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if val_metric<best-1e-10:
            best=val_metric
            best_epoch=epoch
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            wait=0
        else:
            wait+=1

        if wait>=cfg.patience:
            break

    model.load_state_dict(best_state,strict=True)
    return model,best_epoch,pd.DataFrame(history)

def refit_phase_b(combined_data,best_epoch,seed):
    set_seed(seed)

    model=PRRStatReranker().to(DEVICE)
    opt=torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.wd,
    )

    rng=np.random.default_rng(seed+202)

    for _ in range(int(best_epoch)):
        listwise_epoch(
            model,opt,combined_data,rng
        )

    model.eval()
    return model

print("[PASS] training code ready")


[PASS] training code ready


In [9]:

# Cell 9 — retrieval forecast from a trained PRR-Stat model

@torch.no_grad()
def retrieval_forecast(
    dataset,
    h,
    query_anchor,
    query_channel,
    memory_boundary,
    med,
    iqr,
    model,
):
    z=DATA[dataset]["z"]
    N=len(query_anchor)
    M=cfg.top_m

    retrieved=np.empty((N,h),np.float32)
    analog=np.empty(N,np.float32)

    stride=memory_stride(dataset)

    for ch in np.unique(query_channel):
        rows=np.flatnonzero(query_channel==ch)
        qa=query_anchor[rows]

        mem_a=np.arange(
            cfg.retrieval_seq_len,
            int(memory_boundary)-h+1,
            stride,
            dtype=np.int64,
        )

        mem_past=past_windows(z,int(ch),mem_a)
        mem_pat=batch_pattern(mem_past)
        mem_stat=scale_stats(
            context7(mem_past),
            med,
            iqr,
        )
        mem_future=future_residuals(
            z,int(ch),mem_a,h
        )

        q_past=past_windows(z,int(ch),qa)
        q_pat=batch_pattern(q_past)
        q_stat=scale_stats(
            context7(q_past),
            med,
            iqr,
        )
        q_future=future_residuals(
            z,int(ch),qa,h
        )

        ps=q_pat@mem_pat.T

        for local,row in enumerate(rows):
            ix=top_m_indices(
                ps[local],
                M,
            )

            b=ps[local,ix]
            cs=mem_stat[ix]
            qs=q_stat[local]

            qrep=np.repeat(
                qs[None,:],
                M,
                axis=0,
            )

            feat=np.concatenate(
                [
                    b[:,None],
                    qrep,
                    cs,
                    qrep-cs,
                    np.abs(qrep-cs),
                ],
                axis=1,
            ).astype(np.float32)

            score=model(
                torch.from_numpy(
                    feat[None,:,:]
                ).float().to(DEVICE),
                torch.from_numpy(
                    b[None,:].astype(np.float32)
                ).float().to(DEVICE),
            ).cpu().numpy()[0]

            top=np.argsort(
                -score
            )[:cfg.top_k]

            chosen=mem_future[ix[top]]

            qcur=z[int(qa[local])-1,int(ch)]

            retrieved[row]=(
                chosen.mean(axis=0)
                +qcur
            )

            analog[row]=np.mean(
                (
                    chosen
                    -q_future[local][None,:]
                )**2
            )

    return retrieved,analog

print("[PASS] retrieval forecast code ready")


[PASS] retrieval forecast code ready


## 2. Rerun the 16 stride-changed PRR-Stat conditions

In [10]:
# Cell 10 — rerun ONLY the stride-changed PRR-Stat conditions, resumably
#
# Weather/Electricity/Traffic/Solar used stride=24 historically.
# Here they are retrained and reevaluated with stride=8.
#
# ETTh1/Exchange already used stride=8 and are imported later from the old
# validated PRR-Stat table, avoiding unnecessary retraining.

def condition_json(dataset,h):
    return PRRSTAT_ROOT / f"{dataset}_H{h}_result.json"

def condition_npz(dataset,h):
    return PRRSTAT_CACHE_ROOT / f"{dataset}_H{h}_test.npz"

def condition_done(dataset,h):
    return condition_json(dataset,h).exists() and condition_npz(dataset,h).exists()

rerun_rows = []
for dataset in RERUN_DATASETS:
    for h in HORIZONS:
        if condition_done(dataset,h):
            rerun_rows.append(json.loads(condition_json(dataset,h).read_text()))

done_keys = {(r["dataset"], int(r["horizon"])) for r in rerun_rows}

for dataset in RERUN_DATASETS:
    z = DATA[dataset]["z"]
    train_end = DATA[dataset]["train_end"]
    val_end = DATA[dataset]["val_end"]

    for h in HORIZONS:
        if (dataset,h) in done_keys:
            print("[SKIP completed]", dataset, h)
            continue

        print("\n" + "="*110)
        print(f"PRR-Stat stride=8 revalidation: {dataset} H={h}")
        print("="*110)

        med, iqr = fit_feature_scaler(dataset,h)
        initial_boundary = int(cfg.initial_memory_fraction * train_end)

        train_anchor_pool = np.arange(
            max(initial_boundary, cfg.retrieval_seq_len),
            train_end-h+1,
            dtype=np.int64,
        )

        cap = (
            cfg.train_cap_mechanism
            if dataset in ("ETTh1","Weather")
            else cfg.train_cap_frozen
        )

        train_a, train_c = balanced_pairs(
            dataset, train_anchor_pool, cap,
            seed=36000 + 100*HORIZONS.index(h) + DATASETS.index(dataset),
        )
        val_a, val_c = load_manifest(dataset,h,"val")

        t0 = time.perf_counter()
        train_sup = build_supervision(
            dataset,h,train_a,train_c,initial_boundary,med,iqr
        )
        val_sup = build_supervision(
            dataset,h,val_a,val_c,train_end,med,iqr
        )
        print(
            f"Supervision built: train={len(train_a):,}, val={len(val_a):,}, "
            f"sec={time.perf_counter()-t0:.1f}"
        )

        phase_a, best_epoch, hist = fit_phase_a(
            train_sup, val_sup, cfg.seed
        )
        hist.to_csv(
            PRRSTAT_ROOT / f"{dataset}_H{h}_phaseA_history.csv",
            index=False
        )

        val_retrieval, val_analog = retrieval_forecast(
            dataset,h,val_a,val_c,train_end,med,iqr,phase_a
        )

        combined = {
            k: np.concatenate([train_sup[k], val_sup[k]], axis=0)
            for k in (
                "feat","base","target","distance",
                "candidate_anchor","anchor","channel"
            )
        }

        final_model = refit_phase_b(
            combined, best_epoch, cfg.seed
        )

        torch.save(
            {
                "Dataset":dataset,
                "Horizon":h,
                "Seed":cfg.seed,
                "MemoryStride":8,
                "InputDim":29,
                "BestEpoch":int(best_epoch),
                "FeatureMedian":med,
                "FeatureIQR":iqr,
                "StateDict":{
                    k:v.detach().cpu().clone()
                    for k,v in final_model.state_dict().items()
                },
                "Recipe":"PRR-Stat stride-8 revalidation",
            },
            CKPT_ROOT / f"{dataset}_H{h}_PRRStat_stride8_seed0.pt",
        )

        test_a, test_c = load_manifest(dataset,h,"test")
        test_retrieval, test_analog = retrieval_forecast(
            dataset,h,test_a,test_c,val_end,med,iqr,final_model
        )

        row = {
            "dataset":dataset,
            "horizon":int(h),
            "seed":cfg.seed,
            "memory_stride":8,
            "best_epoch":int(best_epoch),
            "val_analog_future_mse":float(val_analog.mean()),
            "test_analog_future_mse":float(test_analog.mean()),
            "pairs":int(len(test_a)),
            "source":"rerun_stride8",
        }

        np.savez_compressed(
            condition_npz(dataset,h),
            anchor=test_a.astype(np.int64),
            channel=test_c.astype(np.int16),
            retrieval=test_retrieval.astype(np.float32),
            analog_mse=test_analog.astype(np.float32),
            memory_boundary=np.asarray(val_end,dtype=np.int64),
            memory_stride=np.asarray(8,dtype=np.int64),
        )
        condition_json(dataset,h).write_text(
            json.dumps(row,indent=2), encoding="utf-8"
        )

        rerun_rows.append(row)
        done_keys.add((dataset,h))

        print(
            f"[DONE] {dataset} H={h}: "
            f"AnalogFutureMSE={row['test_analog_future_mse']:.6f}"
        )

        del train_sup,val_sup,combined,phase_a,final_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

rerun_df = (
    pd.DataFrame(rerun_rows)
    .drop_duplicates(["dataset","horizon"],keep="last")
    .sort_values(["dataset","horizon"])
    .reset_index(drop=True)
)
display(rerun_df)
assert len(rerun_df) == len(RERUN_DATASETS)*len(HORIZONS)
rerun_df.to_csv(PRRSTAT_ROOT/"prr_stat_stride8_rerun_16.csv",index=False)
print("[PASS] changed PRR-Stat conditions =",len(rerun_df))

[SKIP completed] Weather 96
[SKIP completed] Weather 192
[SKIP completed] Weather 336
[SKIP completed] Weather 720
[SKIP completed] Electricity 96
[SKIP completed] Electricity 192
[SKIP completed] Electricity 336
[SKIP completed] Electricity 720
[SKIP completed] Traffic 96
[SKIP completed] Traffic 192
[SKIP completed] Traffic 336
[SKIP completed] Traffic 720
[SKIP completed] Solar 96
[SKIP completed] Solar 192
[SKIP completed] Solar 336
[SKIP completed] Solar 720


,dataset,horizon,seed,memory_stride,best_epoch,val_analog_future_mse,test_analog_future_mse,pairs,source
0,Electricity,96,0,8,15,0.846599,0.875108,8192,rerun_stride8
1,Electricity,192,0,8,19,0.839382,0.896645,8192,rerun_stride8
2,Electricity,336,0,8,10,0.922447,0.930136,8192,rerun_stride8
3,Electricity,720,0,8,23,0.996723,0.976864,8192,rerun_stride8
4,Solar,96,0,8,9,0.404611,0.526962,8192,rerun_stride8
5,Solar,192,0,8,25,0.401322,0.541064,8192,rerun_stride8
6,Solar,336,0,8,19,0.418011,0.569167,8192,rerun_stride8
7,Solar,720,0,8,29,0.388173,0.578178,8192,rerun_stride8
8,Traffic,96,0,8,29,1.356790,1.475479,8192,rerun_stride8
9,Traffic,192,0,8,18,1.523754,1.513514,8192,rerun_stride8


[PASS] changed PRR-Stat conditions = 16


In [11]:
# Cell 11 — rebuild the full 24-condition PRR-Stat table
#
# ETTh1 / Exchange already used memory_stride=8 historically, so their
# frozen values are read from the embedded reviewer-supplementary reference.
# The 16 changed conditions come from the newly rerun stride=8 results.

reused = (
    REFERENCE24[REFERENCE24.dataset.isin(UNCHANGED_DATASETS)]
    [["dataset","horizon","prr_stat_analog_future_mse"]]
    .rename(columns={"prr_stat_analog_future_mse":"test_analog_future_mse"})
    .copy()
)
reused["val_analog_future_mse"] = np.nan
reused["best_epoch"] = np.nan
reused["memory_stride"] = 8
reused["seed"] = 0
reused["pairs"] = np.nan
reused["source"] = "embedded_reference_already_stride8"

new = rerun_df[
    ["dataset","horizon","val_analog_future_mse","test_analog_future_mse",
     "best_epoch","memory_stride","seed","pairs","source"]
].copy()

prr_stat24 = (
    pd.concat([reused,new],ignore_index=True)
    .sort_values(["dataset","horizon"])
    .reset_index(drop=True)
)

assert len(prr_stat24) == 24
assert set(prr_stat24.dataset) == set(DATASETS)
assert (prr_stat24.memory_stride == 8).all()

display(prr_stat24)
prr_stat24.to_csv(PRRSTAT_ROOT/"prr_stat_24conditions_stride8.csv",index=False)
print("[PASS] aligned PRR-Stat table = 24/24, stride=8 everywhere")

,dataset,horizon,test_analog_future_mse,val_analog_future_mse,best_epoch,memory_stride,seed,pairs,source
0,ETTh1,96,1.196500,NaN,NaN,8,0,NaN,embedded_reference_already_stride8
1,ETTh1,192,1.366400,NaN,NaN,8,0,NaN,embedded_reference_already_stride8
2,ETTh1,336,1.486100,NaN,NaN,8,0,NaN,embedded_reference_already_stride8
3,ETTh1,720,1.727200,NaN,NaN,8,0,NaN,embedded_reference_already_stride8
4,Electricity,96,0.875108,0.846599,15.0,8,0,8192.0,rerun_stride8
5,Electricity,192,0.896645,0.839382,19.0,8,0,8192.0,rerun_stride8
6,Electricity,336,0.930136,0.922447,10.0,8,0,8192.0,rerun_stride8
7,Electricity,720,0.976864,0.996723,23.0,8,0,8192.0,rerun_stride8
8,Exchange,96,0.220600,NaN,NaN,8,0,NaN,embedded_reference_already_stride8
9,Exchange,192,0.427600,NaN,NaN,8,0,NaN,embedded_reference_already_stride8


[PASS] aligned PRR-Stat table = 24/24, stride=8 everywhere


In [12]:
# Cell 12 — merge stride-8 PRR-Stat with the existing full PRR frozen reference
#
# Full PRR already used memory_stride=8 for every dataset, so its 24 values
# are taken from the embedded reviewer-supplementary reference and are not rerun.

full_prr = REFERENCE24[
    ["dataset","horizon","prr_analog_future_mse"]
].rename(columns={"prr_analog_future_mse":"test_analog_future_mse"}).copy()

prrstat_vs_prr = prr_stat24.merge(
    full_prr.rename(columns={"test_analog_future_mse":"prr_analog_future_mse"}),
    on=["dataset","horizon"],
    how="inner",
)

prrstat_vs_prr["prr_gain_vs_prr_stat_pct"] = (
    100.0
    * (prrstat_vs_prr.test_analog_future_mse
       - prrstat_vs_prr.prr_analog_future_mse)
    / prrstat_vs_prr.test_analog_future_mse
)

assert len(prrstat_vs_prr) == 24
display(prrstat_vs_prr)
prrstat_vs_prr.to_csv(
    PRRSTAT_ROOT/"prr_stat_vs_full_prr_stride8.csv",
    index=False
)

print(
    "PRR vs PRR-Stat:",
    "wins =", int((prrstat_vs_prr.prr_gain_vs_prr_stat_pct > 0).sum()),
    "/24; mean gain =",
    f"{prrstat_vs_prr.prr_gain_vs_prr_stat_pct.mean():+.3f}%"
)

,dataset,horizon,test_analog_future_mse,val_analog_future_mse,best_epoch,memory_stride,seed,pairs,source,prr_analog_future_mse,prr_gain_vs_prr_stat_pct
0,ETTh1,96,1.196500,NaN,NaN,8,0,NaN,embedded_reference_already_stride8,1.1643,2.691183
1,ETTh1,192,1.366400,NaN,NaN,8,0,NaN,embedded_reference_already_stride8,1.2229,10.502049
2,ETTh1,336,1.486100,NaN,NaN,8,0,NaN,embedded_reference_already_stride8,1.4816,0.302806
3,ETTh1,720,1.727200,NaN,NaN,8,0,NaN,embedded_reference_already_stride8,1.4618,15.365910
4,Electricity,96,0.875108,0.846599,15.0,8,0,8192.0,rerun_stride8,0.7191,17.827330
5,Electricity,192,0.896645,0.839382,19.0,8,0,8192.0,rerun_stride8,0.7345,18.083507
6,Electricity,336,0.930136,0.922447,10.0,8,0,8192.0,rerun_stride8,0.7648,17.775451
7,Electricity,720,0.976864,0.996723,23.0,8,0,8192.0,rerun_stride8,0.8602,11.942667
8,Exchange,96,0.220600,NaN,NaN,8,0,NaN,embedded_reference_already_stride8,0.1652,25.113327
9,Exchange,192,0.427600,NaN,NaN,8,0,NaN,embedded_reference_already_stride8,0.4116,3.741815


PRR vs PRR-Stat: wins = 22 /24; mean gain = +11.454%


## 3. Recompute Pearson with stride 8

In [13]:
# Cell 13 — exact Pearson Top-10 baseline with memory_stride=8 on ALL six datasets
#
# Resumable. This recomputes Pearson under the same historical memory grid used
# by full PRR. No training and no forecaster inference are involved.

def pearson_cache_path(dataset,horizon):
    return PEARSON_CACHE_ROOT / f"{dataset}_H{horizon}_pearson_stride8_test.npz"

def valid_pearson_cache(dataset,horizon):
    p = pearson_cache_path(dataset,horizon)
    if not p.exists():
        return False
    try:
        ma,mc = load_manifest(dataset,horizon,"test")
        with np.load(p,allow_pickle=False) as z:
            a=np.asarray(z["anchor"],np.int64)
            c=np.asarray(z["channel"],np.int64)
            e=np.asarray(z["analog_mse"],np.float32)
            stride=int(np.asarray(z["memory_stride"]).item())
        return (
            np.array_equal(a,ma)
            and np.array_equal(c,mc)
            and e.shape == ma.shape
            and np.isfinite(e).all()
            and stride == 8
        )
    except Exception:
        return False

def pearson_condition_stride8(dataset,horizon,query_chunk=128):
    if valid_pearson_cache(dataset,horizon):
        p=pearson_cache_path(dataset,horizon)
        with np.load(p,allow_pickle=False) as z:
            e=np.asarray(z["analog_mse"],np.float32)
        return {
            "dataset":dataset,
            "horizon":int(horizon),
            "pearson_analog_future_mse":float(e.mean()),
            "pairs":int(len(e)),
            "memory_stride":8,
            "reused":True,
        }

    z = DATA[dataset]["z"]
    boundary = int(DATA[dataset]["val_end"])
    qa,qc = load_manifest(dataset,horizon,"test")
    pair_error = np.full(len(qa),np.nan,dtype=np.float32)

    t0=time.perf_counter()
    for ch in np.unique(qc):
        rows=np.flatnonzero(qc==ch)

        mem_anchor=np.arange(
            cfg.retrieval_seq_len,
            boundary-horizon+1,
            8,
            dtype=np.int64,
        )
        assert len(mem_anchor) >= cfg.top_k
        assert int(mem_anchor.max()) + int(horizon) <= boundary

        mem_past=past_windows(z,int(ch),mem_anchor)
        mem_pattern=batch_pattern(mem_past)
        mem_future=future_residuals(z,int(ch),mem_anchor,horizon)

        for left in range(0,len(rows),int(query_chunk)):
            local_rows=rows[left:left+int(query_chunk)]
            q_a=qa[local_rows]

            q_pattern=batch_pattern(past_windows(z,int(ch),q_a))
            q_future=future_residuals(z,int(ch),q_a,horizon)

            score=q_pattern @ mem_pattern.T
            top=np.argpartition(
                -score,
                cfg.top_k-1,
                axis=1
            )[:,:cfg.top_k]

            chosen=mem_future[top]
            e=((chosen-q_future[:,None,:])**2).mean(axis=(1,2))
            pair_error[local_rows]=e.astype(np.float32)

        print(
            f"  {dataset:11s} H={horizon:3d} ch={int(ch):3d} "
            f"pairs={len(rows):4d} memory={len(mem_anchor):5d}"
        )

    assert np.isfinite(pair_error).all()
    p=pearson_cache_path(dataset,horizon)
    np.savez_compressed(
        p,
        anchor=qa.astype(np.int64),
        channel=qc.astype(np.int16),
        analog_mse=pair_error.astype(np.float32),
        memory_boundary=np.asarray(boundary,dtype=np.int64),
        memory_stride=np.asarray(8,dtype=np.int64),
        top_k=np.asarray(cfg.top_k,dtype=np.int64),
        retrieval_name=np.asarray("Pearson_centered_cosine_stride8"),
    )

    return {
        "dataset":dataset,
        "horizon":int(horizon),
        "pearson_analog_future_mse":float(pair_error.mean()),
        "pairs":int(len(pair_error)),
        "memory_stride":8,
        "reused":False,
        "seconds":float(time.perf_counter()-t0),
    }

pearson_rows=[]
for dataset in DATASETS:
    for h in HORIZONS:
        print("\n"+"="*90)
        print(f"Pearson stride=8: {dataset} H={h}")
        print("="*90)
        row=pearson_condition_stride8(dataset,h)
        pearson_rows.append(row)
        print("AnalogFutureMSE =",f"{row['pearson_analog_future_mse']:.6f}")

pearson24=(
    pd.DataFrame(pearson_rows)
    .sort_values(["dataset","horizon"])
    .reset_index(drop=True)
)
assert len(pearson24)==24
display(pearson24)
pearson24.to_csv(
    PEARSON_ROOT/"pearson_24conditions_stride8.csv",
    index=False
)
print("[PASS] Pearson stride=8 = 24/24")


Pearson stride=8: ETTh1 H=96
  ETTh1       H= 96 ch=  0 pairs=1170 memory= 1417
  ETTh1       H= 96 ch=  1 pairs=1170 memory= 1417
  ETTh1       H= 96 ch=  2 pairs=1170 memory= 1417
  ETTh1       H= 96 ch=  3 pairs=1170 memory= 1417
  ETTh1       H= 96 ch=  4 pairs=1170 memory= 1417
  ETTh1       H= 96 ch=  5 pairs=1171 memory= 1417
  ETTh1       H= 96 ch=  6 pairs=1171 memory= 1417
AnalogFutureMSE = 1.612267

Pearson stride=8: ETTh1 H=192
  ETTh1       H=192 ch=  0 pairs=1170 memory= 1405
  ETTh1       H=192 ch=  1 pairs=1170 memory= 1405
  ETTh1       H=192 ch=  2 pairs=1170 memory= 1405
  ETTh1       H=192 ch=  3 pairs=1170 memory= 1405
  ETTh1       H=192 ch=  4 pairs=1170 memory= 1405
  ETTh1       H=192 ch=  5 pairs=1171 memory= 1405
  ETTh1       H=192 ch=  6 pairs=1171 memory= 1405
AnalogFutureMSE = 1.817531

Pearson stride=8: ETTh1 H=336
  ETTh1       H=336 ch=  0 pairs=1170 memory= 1387
  ETTh1       H=336 ch=  1 pairs=1170 memory= 1387
  ETTh1       H=336 ch=  2 pairs=1170 

,dataset,horizon,pearson_analog_future_mse,pairs,memory_stride,reused,seconds
0,ETTh1,96,1.612267,8192,8,False,0.189519
1,ETTh1,192,1.817531,8192,8,False,0.209324
2,ETTh1,336,1.992872,8192,8,False,0.243008
3,ETTh1,720,2.254092,8192,8,False,0.319527
4,Electricity,96,1.179589,8192,8,False,1.875920
5,Electricity,192,1.195698,8192,8,False,2.589303
6,Electricity,336,1.241844,8192,8,False,3.365871
7,Electricity,720,1.287729,8192,8,False,5.785326
8,Exchange,96,0.213266,8192,8,False,0.120551
9,Exchange,192,0.423408,8192,8,False,0.137628


[PASS] Pearson stride=8 = 24/24


## 4. Rebuild headline statistics

In [14]:
# Cell 14 — rebuild the paper's aligned 3-way retrieval table and headline statistics

table = (
    pearson24[["dataset","horizon","pearson_analog_future_mse"]]
    .merge(
        prr_stat24[["dataset","horizon","test_analog_future_mse"]]
        .rename(columns={"test_analog_future_mse":"prr_stat_analog_future_mse"}),
        on=["dataset","horizon"],
        how="inner",
    )
    .merge(
        full_prr[["dataset","horizon","test_analog_future_mse"]]
        .rename(columns={"test_analog_future_mse":"prr_analog_future_mse"}),
        on=["dataset","horizon"],
        how="inner",
    )
)

assert len(table)==24

table["prr_stat_gain_vs_pearson_pct"] = (
    100*(table.pearson_analog_future_mse-table.prr_stat_analog_future_mse)
    /table.pearson_analog_future_mse
)
table["prr_gain_vs_pearson_pct"] = (
    100*(table.pearson_analog_future_mse-table.prr_analog_future_mse)
    /table.pearson_analog_future_mse
)
table["prr_gain_vs_prr_stat_pct"] = (
    100*(table.prr_stat_analog_future_mse-table.prr_analog_future_mse)
    /table.prr_stat_analog_future_mse
)

def wtl(x,tol=1e-12):
    x=np.asarray(x,float)
    return {
        "W":int((x>tol).sum()),
        "T":int((np.abs(x)<=tol).sum()),
        "L":int((x<-tol).sum()),
    }

summary=[]
for name,col in [
    ("PRR-Stat vs Pearson","prr_stat_gain_vs_pearson_pct"),
    ("PRR vs Pearson","prr_gain_vs_pearson_pct"),
    ("PRR vs PRR-Stat","prr_gain_vs_prr_stat_pct"),
]:
    x=table[col]
    summary.append({
        "comparison":name,
        **wtl(x),
        "mean_gain_pct":float(x.mean()),
        "median_gain_pct":float(x.median()),
    })

summary_df=pd.DataFrame(summary)

display(table)
display(summary_df)

table.to_csv(REVAL_ROOT/"table1_stride8_revalidated.csv",index=False)
summary_df.to_csv(REVAL_ROOT/"table1_stride8_summary.csv",index=False)

final = {
    "protocol":"all methods use memory_stride=8",
    "conditions":24,
    "prr_stat_vs_pearson":summary[0],
    "prr_vs_pearson":summary[1],
    "prr_vs_prr_stat":summary[2],
}
(REVAL_ROOT/"stride8_revalidation_summary.json").write_text(
    json.dumps(final,indent=2),encoding="utf-8"
)

print(json.dumps(final,indent=2))

,dataset,horizon,pearson_analog_future_mse,prr_stat_analog_future_mse,prr_analog_future_mse,prr_stat_gain_vs_pearson_pct,prr_gain_vs_pearson_pct,prr_gain_vs_prr_stat_pct
0,ETTh1,96,1.612267,1.196500,1.1643,25.787727,27.784915,2.691183
1,ETTh1,192,1.817531,1.366400,1.2229,24.821070,32.716398,10.502049
2,ETTh1,336,1.992872,1.486100,1.4816,25.429221,25.655026,0.302806
3,ETTh1,720,2.254092,1.727200,1.4618,23.374918,35.149060,15.365910
4,Electricity,96,1.179589,0.875108,0.7191,25.812412,39.038078,17.827330
5,Electricity,192,1.195698,0.896645,0.7345,25.010785,38.571465,18.083507
6,Electricity,336,1.241844,0.930136,0.7648,25.100417,38.414156,17.775451
7,Electricity,720,1.287729,0.976864,0.8602,24.140589,33.200226,11.942667
8,Exchange,96,0.213266,0.220600,0.1652,-3.438775,22.538143,25.113327
9,Exchange,192,0.423408,0.427600,0.4116,-0.989983,2.788875,3.741815


,comparison,W,T,L,mean_gain_pct,median_gain_pct
0,PRR-Stat vs Pearson,20,0,4,17.665645,23.828676
1,PRR vs Pearson,23,0,1,26.682667,27.112056
2,PRR vs PRR-Stat,22,0,2,11.453919,7.627321


{
  "protocol": "all methods use memory_stride=8",
  "conditions": 24,
  "prr_stat_vs_pearson": {
    "comparison": "PRR-Stat vs Pearson",
    "W": 20,
    "T": 0,
    "L": 4,
    "mean_gain_pct": 17.66564481485097,
    "median_gain_pct": 23.82867595445449
  },
  "prr_vs_pearson": {
    "comparison": "PRR vs Pearson",
    "W": 23,
    "T": 0,
    "L": 1,
    "mean_gain_pct": 26.682666956794517,
    "median_gain_pct": 27.11205555660254
  },
  "prr_vs_prr_stat": {
    "comparison": "PRR vs PRR-Stat",
    "W": 22,
    "T": 0,
    "L": 2,
    "mean_gain_pct": 11.453919381125758,
    "median_gain_pct": 7.627320752366359
  }
}


In [15]:
# Cell 15 — compare new stride-8 values against the original frozen reference

oldt = REFERENCE24.rename(columns={
    "pearson_analog_future_mse":"old_pearson",
    "prr_stat_analog_future_mse":"old_prr_stat",
    "prr_analog_future_mse":"old_prr",
}).copy()

comp = table.merge(
    oldt[["dataset","horizon","old_pearson","old_prr_stat","old_prr"]],
    on=["dataset","horizon"],
    how="left",
)

comp["delta_pearson"] = comp.pearson_analog_future_mse-comp.old_pearson
comp["delta_prr_stat"] = comp.prr_stat_analog_future_mse-comp.old_prr_stat
comp["delta_prr"] = comp.prr_analog_future_mse-comp.old_prr

display(comp)
comp.to_csv(REVAL_ROOT/"old_vs_stride8_condition_deltas.csv",index=False)

print("\nExpected audit:")
print(" - full PRR delta should be exactly 0 (same frozen full-PRR reference).")
print(" - ETTh1/Exchange PRR-Stat delta should be 0 (already stride=8).")
print(" - Weather/Electricity/Traffic/Solar Pearson and PRR-Stat may change.")

,dataset,horizon,pearson_analog_future_mse,prr_stat_analog_future_mse,prr_analog_future_mse,prr_stat_gain_vs_pearson_pct,prr_gain_vs_pearson_pct,prr_gain_vs_prr_stat_pct,old_pearson,old_prr_stat,old_prr,delta_pearson,delta_prr_stat,delta_prr
0,ETTh1,96,1.612267,1.196500,1.1643,25.787727,27.784915,2.691183,1.6123,1.1965,1.1643,-0.000033,0.000000,0.0
1,ETTh1,192,1.817531,1.366400,1.2229,24.821070,32.716398,10.502049,1.8175,1.3664,1.2229,0.000031,0.000000,0.0
2,ETTh1,336,1.992872,1.486100,1.4816,25.429221,25.655026,0.302806,1.9929,1.4861,1.4816,-0.000028,0.000000,0.0
3,ETTh1,720,2.254092,1.727200,1.4618,23.374918,35.149060,15.365910,2.2541,1.7272,1.4618,-0.000008,0.000000,0.0
4,Electricity,96,1.179589,0.875108,0.7191,25.812412,39.038078,17.827330,2.6259,1.9919,0.7191,-1.446311,-1.116792,0.0
5,Electricity,192,1.195698,0.896645,0.7345,25.010785,38.571465,18.083507,2.6210,2.0097,0.7345,-1.425302,-1.113055,0.0
6,Electricity,336,1.241844,0.930136,0.7648,25.100417,38.414156,17.775451,2.6610,2.0390,0.7648,-1.419156,-1.108864,0.0
7,Electricity,720,1.287729,0.976864,0.8602,24.140589,33.200226,11.942667,2.8247,2.1850,0.8602,-1.536971,-1.208136,0.0
8,Exchange,96,0.213266,0.220600,0.1652,-3.438775,22.538143,25.113327,0.2133,0.2206,0.1652,-0.000034,0.000000,0.0
9,Exchange,192,0.423408,0.427600,0.4116,-0.989983,2.788875,3.741815,0.4234,0.4276,0.4116,0.000008,0.000000,0.0



Expected audit:
 - full PRR delta should be exactly 0 (same frozen full-PRR reference).
 - ETTh1/Exchange PRR-Stat delta should be 0 (already stride=8).
 - Weather/Electricity/Traffic/Solar Pearson and PRR-Stat may change.


In [16]:
# Cell 16 — compact paper-ready table

paper = table.copy()
for col in [
    "pearson_analog_future_mse",
    "prr_stat_analog_future_mse",
    "prr_analog_future_mse",
]:
    paper[col]=paper[col].map(lambda x:f"{x:.4f}")

for col in [
    "prr_stat_gain_vs_pearson_pct",
    "prr_gain_vs_pearson_pct",
    "prr_gain_vs_prr_stat_pct",
]:
    paper[col]=paper[col].map(lambda x:f"{x:+.2f}%")

paper=paper.rename(columns={
    "dataset":"Dataset",
    "horizon":"H",
    "pearson_analog_future_mse":"Pearson",
    "prr_stat_analog_future_mse":"PRR-Stat (ablation)",
    "prr_analog_future_mse":"PRR (main)",
    "prr_stat_gain_vs_pearson_pct":"PRR-Stat vs Pearson",
    "prr_gain_vs_pearson_pct":"PRR vs Pearson",
    "prr_gain_vs_prr_stat_pct":"PRR vs PRR-Stat",
})

display(paper)
paper.to_csv(REVAL_ROOT/"paper_table1_stride8_revalidated.csv",index=False)

print("\nSaved under:",REVAL_ROOT)
print("Key files:")
print(" - table1_stride8_revalidated.csv")
print(" - table1_stride8_summary.csv")
print(" - stride8_revalidation_summary.json")
print(" - paper_table1_stride8_revalidated.csv")
print("\nIf the headline statistics change, use these files to update the manuscript and supplementary material.")

,Dataset,H,Pearson,PRR-Stat (ablation),PRR (main),PRR-Stat vs Pearson,PRR vs Pearson,PRR vs PRR-Stat
0,ETTh1,96,1.6123,1.1965,1.1643,+25.79%,+27.78%,+2.69%
1,ETTh1,192,1.8175,1.3664,1.2229,+24.82%,+32.72%,+10.50%
2,ETTh1,336,1.9929,1.4861,1.4816,+25.43%,+25.66%,+0.30%
3,ETTh1,720,2.2541,1.7272,1.4618,+23.37%,+35.15%,+15.37%
4,Electricity,96,1.1796,0.8751,0.7191,+25.81%,+39.04%,+17.83%
5,Electricity,192,1.1957,0.8966,0.7345,+25.01%,+38.57%,+18.08%
6,Electricity,336,1.2418,0.9301,0.7648,+25.10%,+38.41%,+17.78%
7,Electricity,720,1.2877,0.9769,0.8602,+24.14%,+33.20%,+11.94%
8,Exchange,96,0.2133,0.2206,0.1652,-3.44%,+22.54%,+25.11%
9,Exchange,192,0.4234,0.4276,0.4116,-0.99%,+2.79%,+3.74%



Saved under: /data/code/which_histories_matter_ext/results/stride8_revalidation
Key files:
 - table1_stride8_revalidated.csv
 - table1_stride8_summary.csv
 - stride8_revalidation_summary.json
 - paper_table1_stride8_revalidated.csv

If the headline statistics change, use these files to update the manuscript and supplementary material.
